# Predicta V4 概率验证（BTC 5m/15m）

本 Notebook 对 Pine 指标 **Predicta Futures - Next Candle Predictor V4** 进行 Python 复刻验证，目标包括：

1. 评估“下一根 K 线涨跌”方向预测效果；
2. 检验 `longPct/shortPct` 是否可作为校准概率；
3. 对比 Isotonic 与 Platt 后校准的改进幅度。

In [ ]:
from pathlib import Path
import json
import pandas as pd

import sys
sys.path.append(str(Path('..').resolve() / 'src'))

from predicta_v4 import PredictaConfig, compute_predicta_v4, download_binance_spot_ohlcv, split_train_valid
from evaluation import summarize_all, evaluate_by_group, fit_calibrators, apply_calibration


In [ ]:
root = Path('..').resolve()
data_dir = root / 'data'
outputs_dir = root / 'outputs'
reports_dir = root / 'reports'
for d in [data_dir, outputs_dir, reports_dir]:
    d.mkdir(parents=True, exist_ok=True)

cfg = PredictaConfig()
timeframes = ['5m', '15m']


In [ ]:
all_summary = []

for tf in timeframes:
    csv_path = data_dir / f'btc_usdt_{tf}.csv'
    if csv_path.exists():
        raw = pd.read_csv(csv_path)
        raw['timestamp'] = pd.to_datetime(raw['timestamp'], utc=True)
    else:
        raw = download_binance_spot_ohlcv(
            symbol='BTC/USDT', timeframe=tf, start_utc='2024-01-01', out_csv=str(csv_path)
        )

    feat = compute_predicta_v4(raw, cfg)
    valid = feat[feat['is_valid'] & feat['next_close'].notna()].copy()
    train, test = split_train_valid(valid, train_ratio=0.7)

    baseline = summarize_all(test)
    by_regime = evaluate_by_group(test, 'vol_regime')

    cal = fit_calibrators(train)
    test_c = apply_calibration(test, cal)
    test_c['pred_dir_iso'] = (test_c['long_prob_iso'] >= 0.5).astype(int)
    test_c['pred_dir_platt'] = (test_c['long_prob_platt'] >= 0.5).astype(int)

    iso = summarize_all(test_c, prob_col='long_prob_iso', pred_col='pred_dir_iso')
    platt = summarize_all(test_c, prob_col='long_prob_platt', pred_col='pred_dir_platt')

    out = {
        'timeframe': tf,
        'n_test': len(test),
        'raw': {'direction': baseline['direction'], 'calibration': baseline['calibration']},
        'isotonic': {'direction': iso['direction'], 'calibration': iso['calibration']},
        'platt': {'direction': platt['direction'], 'calibration': platt['calibration']},
    }
    all_summary.append(out)

    by_regime.to_csv(outputs_dir / f'direction_by_regime_{tf}.csv', index=False)
    with open(outputs_dir / f'summary_{tf}.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

all_summary


## 结论撰写建议

- 先看方向指标（Accuracy/MCC/AUC）是否显著优于基线；
- 再看 ECE/Brier/LogLoss 与 reliability 曲线；
- 若原始校准较差但后校准提升明显，则 `longPct` 更适合作为排序分数，经校准后再用于仓位决策。